In [ ]:
# CELL 2 - Start here after restart
!pip install transformers==4.35.2 datasets==2.14.6 accelerate==0.25.0 sentencepiece -q
!pip install evaluate==0.4.1 jiwer==3.0.3 sacrebleu==2.3.1 -q

import os
os.environ["WANDB_DISABLED"] = "true"

print("✓ Packages installed successfully!")

✓ Packages installed successfully!


In [ ]:
!pip install peft==0.6.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.7/174.7 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1


In [ ]:
import pandas as pd
import re
import random

# 1. File
# Reading the source CSV and converting the 'Clean' column into a list for processing
df_base = pd.read_csv('a01_1720(in).csv')
clean_list = df_base['Clean'].dropna().tolist()

def generate_variations(text):
    results = set()
    text = text.strip()

    # Helper Functions
    def apply_grammar_noise(t):
        # Simulating common ASR errors like mixing up nasal sounds (Anusvar) and Visarg
        t = t.replace("हैं", random.choice(["है", "हे"]))
        t = t.replace("मैं", "में")
        t = t.replace("हूँ", "हु")
        t = t.replace("अतः", "अत").replace("पुनः", "पुन").replace("संभवतः", "संभवत")
        return t

    def strip_dash(t):
      # Removing hyphens to simulate noise in compound words
        return t.replace("-", " ")

    # Identifying Paragraphs
    # Determining if the input is a paragraph based on multiple sentence endings
    sentence_ends = re.findall(r'[।?|]', text)
    is_paragraph = len(sentence_ends) > 1

    # Applying grammatical errors to every variant
    base_noisy_text = apply_grammar_noise(text)

    # Version 1: Extreme Noise (No Punctuation + No Dash + Grammar Error)
    extreme_noisy = re.sub(r'[!?,।|?]', '', strip_dash(base_noisy_text))
    results.add(" ".join(extreme_noisy.split()))

    #Punctuation Logic
    #Identifying which specific punctuation marks are present in the current text
    marks_to_check = ['!', ',', '।', '?', '|']
    present = [m for m in marks_to_check if m in text]

    if is_paragraph:
        # PARAGRAPH SPECIFIC RULES
        # Rule A: Removing only the boundary markers (sentence endings)
        results.add(re.sub(r'[।?|]', '', base_noisy_text))

        # Rule B: Removing internal markers like commas but keeping the sentence endings
        results.add(re.sub(r'[,!]', '', base_noisy_text))

        # Rule C: Merging sentences by removing only the first occurrence of a punctuation mark
        for mark in set(sentence_ends):
            results.add(base_noisy_text.replace(mark, " ", 1)) # Sirf pehla occurrence hataya

    else:
        #NORMAL SENTENCE RULES (9 Rules)
        #Logic for handling sentences with single, double, or multiple punctuation marks
        if len(present) == 1:
            results.add(text.replace(present[0], ""))

        elif len(present) == 2:
            m1, m2 = present[0], present[1]
            results.add(text.replace(m1, "")) # Variation without first mark
            results.add(text.replace(m2, "")) # Variation without second mark
            results.add(re.sub(r'[!?,।|?]', '', text)) # Variation without any marks
        elif len(present) >= 3:
            for m in present:
                results.add(text.replace(m, ""))
            results.add(re.sub(r'[!?,]', '', text)) # Keeping only sentence boundaries
            results.add(re.sub(r'[।?|]', '', text))# Keeping only internal punctuations
    # Dash Logic
    # Handling words like 'धीरे-धीरे' by creating variations with and without hyphens
    if "-" in text:
        no_dash = strip_dash(text)
        results.add(no_dash)
        results.add(re.sub(r'[!?,।|?]', '', no_dash))

    return results

# 2. Pairs Generatation
final_pairs = []

for sentence in clean_list:
    # Including the 'Clean-to-Clean' pair to ensure the model learns to leave correct text as is
    final_pairs.append({"Noisy": sentence, "Clean": sentence})

   # Generating and collecting all unique noisy versions for each clean sentence
    noisy_variants = generate_variations(sentence)
    for nv in noisy_variants:
        nv_clean = " ".join(nv.split()) # Normalizing whitespace
        if nv_clean != sentence.strip():
            final_pairs.append({"Noisy": nv_clean, "Clean": sentence})

# 3. Save
#Dropping any identical pairs and saving the final expanded dataset
df_final = pd.DataFrame(final_pairs).drop_duplicates()
df_final.to_csv('augmenteda02.csv', index=False, encoding='utf-8-sig')

print(f"Success! Total Rows: {len(df_final)}")

Success! Total Rows: 6759


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

# 1.Loading the Augmented Base Dataset(6726 pairs)
old_df = pd.read_csv('augmenteda02.csv')

# 2. Targeted Data (1001 pairs) to fix specific failures
targeted_df = pd.read_csv('targa03_1000(in).csv')

# 3. We repeat the targeted samples twice (1000 * 2 = 2000 rows)
#This ensures these critical patterns are seen more frequently by the model
targeted_upsampled = pd.concat([targeted_df] * 2, ignore_index=True)

# 4. Merge and Shuffle
#Combining base data with the doubled targeted data for a balanced training pool
## Total Rows: 6,758 + 2000 = 8,758
df = pd.concat([old_df, targeted_upsampled], ignore_index=True)
df = shuffle(df, random_state=42).reset_index(drop=True)

def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).strip()
    return ' '.join(text.split())

df['Noisy'] = df['Noisy'].apply(clean_text)
df['Clean'] = df['Clean'].apply(clean_text)
df = df[(df['Noisy'] != "") & (df['Clean'] != "")]

# IndicBART Formatting
# Adding special language and separator tokens required by IndicBART
df['input_text'] = df['Noisy'] + " </s> <2hi>"
df['target_text'] = "<2hi> " + df['Clean'] + " </s>"

# Split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
test_df.to_csv('testdata04.csv', index=False, encoding='utf-8-sig')
print(f"Dataset Ready! Total Training Data: {len(train_df)},\n Total testing Data: {len(test_df)}")


Dataset Ready! Total Training Data: 7005,
 Total testing Data: 1752


In [ ]:
from transformers import AutoTokenizer, MBartForConditionalGeneration
import torch

model_name = "ai4bharat/IndicBART"

# tokenizer configuration
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    do_lower_case=False,
    use_fast=False,
    keep_accents=True
)

# MBart model (not Auto)
model = MBartForConditionalGeneration.from_pretrained(model_name)

# Stability fix
model = model.float()

# Move to device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f" IndicBART loaded on {device}")

# Model info
total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params:,}")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `for

 IndicBART loaded on cuda
  Total parameters: 244,017,152


In [ ]:
from datasets import Dataset

# Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df[['input_text', 'target_text']])
test_dataset = Dataset.from_pandas(test_df[['input_text', 'target_text']])

print(f" Datasets created!")
print(f"  train_dataset: {len(train_dataset):,} samples")
print(f"  test_dataset:  {len(test_dataset):,} samples")

 Datasets created!
  train_dataset: 7,005 samples
  test_dataset:  1,752 samples


In [ ]:
def tokenize_fn(examples):
    """
    Tokenize with IndicBART special tokens
    Max length 200 for long paragraphs
    """
    # Tokenize inputs
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=200,
        padding="max_length",
        truncation=True
    )

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["target_text"],
            max_length=200,
            padding="max_length",
            truncation=True
        )

    # Replace pad tokens with -100 (ignored in loss calculation)
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label_seq]
        for label_seq in labels["input_ids"]
    ]

    return model_inputs

# Apply tokenization
print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

from datasets import DatasetDict
tokenized_datasets = DatasetDict({
    "train": tokenized_train,
    "test": tokenized_test
})

print(f" Tokenization completed with max_length=200")
print(f"  Train: {len(tokenized_datasets['train']):,} samples")
print(f"  Test:  {len(tokenized_datasets['test']):,} samples")

Tokenizing datasets...


Map:   0%|          | 0/7005 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:3856: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1752 [00:00<?, ? examples/s]

 Tokenization completed with max_length=200
  Train: 7,005 samples
  Test:  1,752 samples


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Training Arguments
args = Seq2SeqTrainingArguments(
    output_dir="./indicbart-hindi-asr-fix",

    # Evaluation and saving
    evaluation_strategy="epoch",
    save_strategy="epoch",

    # Batch configuration
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # Effective batch = 4 * 8 = 32

    # Learning
    learning_rate=2e-5,
    num_train_epochs=10,
    warmup_steps=500,
    weight_decay=0.01,

    # Generation
    predict_with_generate=True,

    # Performance
    fp16=False,  # Keep False to prevent NaN loss

    # Logging
    logging_dir="./logs",
    logging_steps=100,

    # Checkpointing
    save_total_limit=3,

    # Reporting
    report_to="none"
)

print(" Training configuration created")
print(f"  Effective batch size: {args.per_device_train_batch_size * args.gradient_accumulation_steps}")
print(f"  Epochs: {args.num_train_epochs}")
print(f"  FP16: {args.fp16}")

# Data Collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

print("Training configuration created")

 Training configuration created
  Effective batch size: 32
  Epochs: 10
  FP16: False
Training configuration created


In [ ]:
# trainer WITH metrics
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Trainer created")
print(f"\nTraining will start with:")
print(f"  Train samples: {len(tokenized_train):,}")
print(f"  Eval samples: {len(tokenized_test):,}")

# Start training
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)

trainer.train()

print("\n Training completed!")

# Save final model
trainer.save_model("./final02_model")
tokenizer.save_pretrained("./final02_model")
print(" Model saved to ./final02_model/")

Trainer created

Training will start with:
  Train samples: 7,005
  Eval samples: 1,752

STARTING TRAINING


Epoch,Training Loss,Validation Loss
1,2.801200,2.198112
2,1.284800,0.866690
3,0.707000,0.447145
4,0.416500,0.205992
5,0.243600,0.132840
6,0.153300,0.113581
7,0.134800,0.107177
8,0.126200,0.102661
9,0.122500,0.100347


Epoch,Training Loss,Validation Loss
1,2.801200,2.198112
2,1.284800,0.866690
3,0.707000,0.447145
4,0.416500,0.205992
5,0.243600,0.132840
6,0.153300,0.113581
7,0.134800,0.107177
8,0.126200,0.102661
9,0.122500,0.100347
10,0.114900,0.099930



 Training completed!
 Model saved to ./final02_model/


In [ ]:
import torch
from transformers import MBartForConditionalGeneration, AutoTokenizer

# 1. Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def normalize_hindi_text(noisy_text):
    # Prepare input with the same format used in training
    input_text = noisy_text + " </s> <2hi>"
    inputs = tokenizer(input_text, return_tensors="pt", padding=True).to(device)

    # Generate with corrected length and beam search
    with torch.no_grad():
        output_tokens = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=200,
            num_beams=5,             # Better quality
            early_stopping=True,
            forced_bos_token_id=tokenizer.encode("<2hi>")[0]
        )

    # Decode and clean up
    decoded = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    return decoded.replace("<2hi>", "").strip()

# 2. Test Cases
test_samples = [
    "आज मोसम अछा है", # Spelling/Grammar noise
    "वहा धीरे धीरे चल रहा था।", # Punctuation/Dash noise
    "क्या आप बता सकते हें की स्टेशन कहा है?" # Complex question noise
]

print("="*50)
print("INDICBART HINDI NORMALIZER TEST")
print("="*50)

for noisy in test_samples:
    clean = normalize_hindi_text(noisy)
    print(f"NOISY: {noisy}")
    print(f"CLEAN: {clean}")
    print("-" * 30)

INDICBART HINDI NORMALIZER TEST
NOISY: आज मोसम अछा है
CLEAN: आज मोसम अछा है।
------------------------------
NOISY: वहा धीरे धीरे चल रहा था।
CLEAN: वहा धीरे-धीरे चल रहा था।
------------------------------
NOISY: क्या आप बता सकते हें की स्टेशन कहा है?
CLEAN: क्या आप बता सकते हैं की स्टेशन कहाँ है?
------------------------------


In [ ]:
test = [

    "में जानता हु की में वहा नहीं था",
    "में बाज़ार में था मेंने तुम्हे नहीं देखा",
    "रुको कहा जा रहे हो",

    # Cases (Single & Multi-sentence)
    "क्या तुम जानते हो की वो कहा रहता है मुझे बताओ",
    "कल बारिश होगी क्या तुम्हे पता है",
    "नमस्ते आप बहुत दिनो बाद मिले",
    "वहा बहुत भीड थी मे डरा हुआ था",
    "ये काम मुझे समझ नही आ रहा हे",
    "जल्दी चलो ट्रेन छूट जाएगी देर हो रही है",



    "कल छुट्टी है क्या तुम घर आओगे",
    "क्या तुम पागल हो ऐसा कोन करता है",
    "खाना तैयार है जल्दी आओ वरना ठंडा हो जाएगा",
    "नमस्ते सर में कल बाज़ार गया था वहा मेने देखा की बहुत भीड है पर मुझे समझ नही आया की लोग इतने परेशां क्यूँ थे क्या आपको पता है की वहा क्या हुआ था अगर आपको कुछ खबर मिले तो मुझे बताना में इंतज़ार करुगा"
]

print("Running Final Evaluation...\n" + "="*80)

for i, noisy in enumerate(test, 1):
    output = normalize_hindi_text(noisy)
    print(f"Test {i} (Input) : {noisy}")
    print(f"Test {i} (Output): {output}")
    print("-" * 80)

Running Final Evaluation...
Test 1 (Input) : में जानता हु की में वहा नहीं था
Test 1 (Output): मैं जानता हूँ की मैं वहाँ नहीं था।
--------------------------------------------------------------------------------
Test 2 (Input) : में बाज़ार में था मेंने तुम्हे नहीं देखा
Test 2 (Output): मैं बाज़ार में था। मैंने तुम्हें वहाँ नहीं देखा।
--------------------------------------------------------------------------------
Test 3 (Input) : रुको कहा जा रहे हो
Test 3 (Output): रुको! कहाँ जा रहे हो?
--------------------------------------------------------------------------------
Test 4 (Input) : क्या तुम जानते हो की वो कहा रहता है मुझे बताओ
Test 4 (Output): क्या तुम जानते हो की वह कहाँ रहता है? मुझे बताओ।
--------------------------------------------------------------------------------
Test 5 (Input) : कल बारिश होगी क्या तुम्हे पता है
Test 5 (Output): कल बारिश होगी। क्या तुम्हें पता है?
--------------------------------------------------------------------------------
Test 6 (Input) : नमस्ते आप बहुत दिन

In [ ]:
# --- EXTENDED PARAGRAPH & COMPLEX TEST SUITE ---
test = [
    # 1. Official/Professional Paragraph (Testing connectives like 'kyuki', 'isliye')
    "नमस्ते सर में कल दफ्तर नहीं आ पाया क्युकी मेरी तबियत ठीक नहीं थी और में डॉक्टर के पास गया था इसलिए क्या आप मुझे कल की मीटिंग की जानकारी दे सकते है ताकि में अपना काम पूरा कर सकू",

    # 2. Daily Life/Instructional (Testing flow and urgent tone)
    "सावधान आगे रास्ता खराब है वहा बहुत पत्थर और कांच के टुकड़े पड़े है अपनी गाड़ी धीरे चलाइये वरना टायर पंक्चर हो सकता है क्या आपको मेरी बात समझ आ रही है",

    # 3. Emotional/Storytelling (Testing gender consistency and long-term context)
    "मेरी दादी ने कहा की में बहुत बहादुर हु पर में जानता हु की उस रात में बहुत डरा हुआ था वहा अँधेरा था और चारो ओर अजीब सी आवाज़े आ रही थी क्या आप कभी ऐसी स्थिति में फंसे है",

    # 4. Market/Conversation (Testing numeric and 'Me/Main' confusion)
    "बाजार मे बहुत भीड थी मेने १० किलो चीनी और कुछ फल ख़रीदे पर मुझे दुकानदार ने पैसे कम वापस दिए क्युकी उसे लगा की में हिसाब नहीं जानता हु",

    # 5. Travel/Urgency (Testing multiple sentence boundaries)
    "जल्दी चलो भाई ट्रेन छूट जाएगी हमे प्लेटफार्म नंबर ४ पर पहुचना है और हमारे पास सिर्फ ५ मिनट बचे है क्या तुम सारा सामान उठा सकते हो या में किसी की मदद लू",

    # 6. House/Domestic (Testing 'In vs I' and spelling)
    "कमरे मे बहुत धूल जमी है मुजे पूरा बिस्वास है की तुमने खिड़की खुली छोड़ी थी इसलिए अब सारा सामान साफ करो और इसे फिर से जमाओ",

    # 7. Weather/Speculation (Testing questions in middle of paragraphs)
    "कल मौसम बहुत ख़राब था क्या तुम्हे पता है की वहा ओले गिरे है में तो घर के अंदर ही था पर बाहर खड़े पेड़ गिर गए है",

    # 8. Your Original Long Case (For comparison)
    "नमस्ते सर में कल बाज़ार गया था वहा मेने देखा की बहुत भीड है पर मुझे समझ नही आया की लोग इतने परेशां क्यूँ थे क्या आपको पता है की वहा क्या हुआ था अगर आपको कुछ खबर मिले तो मुझे बताना में इंतज़ार करुगा"
]

print("Running Evaluation on Large Paragraphs & Complex Context...\n" + "="*80)

for i, noisy in enumerate(test, 1):
    output = normalize_hindi_text(noisy)
    print(f"Test {i} (Input) : {noisy}")
    print(f"Test {i} (Output): {output}")
    print("-" * 80)

Running Evaluation on Large Paragraphs & Complex Context...
Test 1 (Input) : नमस्ते सर में कल दफ्तर नहीं आ पाया क्युकी मेरी तबियत ठीक नहीं थी और में डॉक्टर के पास गया था इसलिए क्या आप मुझे कल की मीटिंग की जानकारी दे सकते है ताकि में अपना काम पूरा कर सकू
Test 1 (Output): नमस्ते सर! मैं कल दफ्तर नहीं आ पाया क्योंकि मेरी तबियत ठीक नहीं थी और मैं डॉक्टर के पास गया था। इसलिए क्या आप मुझे कल की मीटिंग की जानकारी दे सकते हैं ताकि मैं अपना काम पूरा कर सकूँ?
--------------------------------------------------------------------------------
Test 2 (Input) : सावधान आगे रास्ता खराब है वहा बहुत पत्थर और कांच के टुकड़े पड़े है अपनी गाड़ी धीरे चलाइये वरना टायर पंक्चर हो सकता है क्या आपको मेरी बात समझ आ रही है
Test 2 (Output): सावधान! आगे रास्ता खराब है। वहाँ बहुत पत्थर और कांच के टुकड़े पड़े हैं। अपनी गाड़ी धीरे चलाइये वरना टायर पंक्चर हो सकता है। क्या आपको मेरी बात समझ आ रही है?
--------------------------------------------------------------------------------
Test 3 (Input) : मेरी दादी ने कहा की में बहुत 

In [ ]:
test_cases = [
    "खाना खाया और सो गया",  # Missing subject
    "बाजार गया था मैं",           # Missing subject
    "किताब पढ़ी",             # Missing subject
    "मैं बाजार गया था",       # Has subject already
]

for text in test_cases:
    output = normalize_hindi_text(text)
    print(f"Input:  {text}")
    print(f"Output: {output}")
    print("-" * 50)

Input:  खाना खाया और सो गया
Output: खाना खाया और सो गया।
--------------------------------------------------
Input:  बाजार गया था मैं
Output: बाजार गया था मैं।
--------------------------------------------------
Input:  किताब पढ़ी
Output: किताब पढ़ी।
--------------------------------------------------
Input:  मैं बाजार गया था
Output: मैं बाजार गया था।
--------------------------------------------------


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

# Source:trained model
source_model = "./final02_model"

# Destination: Google Drive
drive_destination = "/content/drive/MyDrive/Hindi_Text_Normalizer_Model02"

# Copy the entire model folder
print("Copying model to Google Drive...")
print("This may take 2-3 minutes...")

if os.path.exists(drive_destination):
    print(f" Removing existing model at {drive_destination}")
    shutil.rmtree(drive_destination)

shutil.copytree(source_model, drive_destination)

print(f"Model saved to Google Drive!")
print(f"Location: {drive_destination}")
print(f"\nFiles saved:")
for file in os.listdir(drive_destination):
    file_path = os.path.join(drive_destination, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  - {file} ({size_mb:.2f} MB)")

Copying model to Google Drive...
This may take 2-3 minutes...
Model saved to Google Drive!
Location: /content/drive/MyDrive/Hindi_Text_Normalizer_Model02

Files saved:
  - config.json (0.00 MB)
  - special_tokens_map.json (0.00 MB)
  - model.safetensors (931.13 MB)
  - generation_config.json (0.00 MB)
  - spiece.model (1.81 MB)
  - added_tokens.json (0.00 MB)
  - tokenizer_config.json (0.00 MB)
  - training_args.bin (0.00 MB)


In [ ]:
import pandas as pd
from tqdm import tqdm

# test_df
test_df = pd.read_csv('testdata04.csv')

print(f" {len(test_df)} test sentences.")

 1752 test sentences.


In [ ]:
from tqdm import tqdm
tqdm.pandas()
#  Processing
print(f" Processing {len(test_df)} sentences...")
test_df['Model_Output'] = test_df['Noisy'].progress_apply(normalize_hindi_text)


 Processing 1752 sentences...


100%|██████████| 1752/1752 [07:25<00:00,  3.94it/s]


In [ ]:
import pandas as pd
import re
from jiwer import wer, cer
from tqdm import tqdm
import numpy as np

def run_clean_final_report(df):
    # Converting the target and predicted text into lists
    targets = df['Target'].astype(str).tolist()
    preds = df['Model_Output'].astype(str).tolist()

    word_overlaps = []
    style_mismatches = 0

    print(f"Final Evaluation on {len(df)} samples...")

    for t, p in tqdm(zip(targets, preds), total=len(targets)):
        # 1. Word Match (Spelling & Vocabulary Focus)
        # Comparing words while ignoring punctuation and symbols
        t_w = set(re.sub(r'[^\u0900-\u097F\s]', '', t).split())
        p_w = set(re.sub(r'[^\u0900-\u097F\s]', '', p).split())

        if t_w:
            word_overlaps.append(len(t_w.intersection(p_w)) / len(t_w))

        # 2. Finding cases where words are correct but formatting (like spaces) differs
        t_clean = "".join(re.findall(r'[\u0900-\u097F0-9]', t))
        p_clean = "".join(re.findall(r'[\u0900-\u097F0-9]', p))
        if t_clean == p_clean and t != p:
            style_mismatches += 1

    # Calculating Final scores
    final_wer = wer(targets, preds)
    final_cer = cer(targets, preds)
    final_word_acc = np.mean(word_overlaps) * 100

    # Displaying the final performance report
    print("\n" + "═"*60)
    print(" FINAL ASR NORMALIZATION REPORT")
    print("═"*60)
    print(f" Word-Level Accuracy (ignores punctuation to check contextual correction): {final_word_acc:.2f}%")
    print(f" Word Error Rate (WER):           {final_wer:.4f}")
    print(f" Character Error Rate (CER):      {final_cer:.4f}")
    print(f" Formatting Only Mismatches:      {style_mismatches} sentences")
    print("═"*60)
    print("\n NOTE: Word-Level Accuracy measures contextual word identification.")
    print(" NOTE: Formatting mismatches do not affect the linguistic quality.")

    return {
        "Word_Acc": final_word_acc,
        "WER": final_wer,
        "CER": final_cer,
        "Style_Issues": style_mismatches
    }

# EXECUTE
final_metrics = run_clean_final_report(test_df.rename(columns={'Clean': 'Target'}))

Final Evaluation on 1752 samples...


100%|██████████| 1752/1752 [00:00<00:00, 26117.32it/s]


════════════════════════════════════════════════════════════
 FINAL ASR NORMALIZATION REPORT
════════════════════════════════════════════════════════════
 Word-Level Accuracy (ignores punctuation to check contextual correction): 96.96%
 Word Error Rate (WER):           0.0356
 Character Error Rate (CER):      0.0129
 Formatting Only Mismatches:      101 sentences
════════════════════════════════════════════════════════════

 NOTE: Word-Level Accuracy measures contextual word identification.
 NOTE: Formatting mismatches do not affect the linguistic quality.


In [ ]:
# @title
!pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 41.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=1bbe8dd523264a4d5ebdc9f33fddec27c46b9f6b15c5f7dc6bbee6c7ec90428e
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
!pip install SpeechRecognition pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 31.5 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import speech_recognition as sr
from pydub import AudioSegment
import os
import uuid

# FUNCTION 1: Manual Text Normalization
def manual_text_normalize(noisy_text):
    if not noisy_text.strip():
        return "Please enter some text."
    # Calling inference function
    return normalize_hindi_text(noisy_text)

# FUNCTION 2: Audio Pipeline
def process_audio_pipeline(audio_path):
    if audio_path is None or not os.path.exists(audio_path):
        return "Audio signal not detected.", ""
#to process audio data
    recognizer = sr.Recognizer()
    unique_wav = f"temp_{uuid.uuid4().hex}.wav"

    try:
        audio = AudioSegment.from_file(audio_path)
        audio.export(unique_wav, format="wav") #loaded audio saved on disk

        with sr.AudioFile(unique_wav) as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio_data = recognizer.record(source)
            raw_text = recognizer.recognize_google(audio_data, language="hi-IN")

        if raw_text.strip():
            # Using model function
            clean_text = normalize_hindi_text(raw_text)
            return raw_text, clean_text
        else:
            return "Speech not recognized.", "Please speak clearly."

    except Exception as e:
        return f"Error: {str(e)}", "Please try again."
    finally:
        if os.path.exists(unique_wav):
            os.remove(unique_wav)

# GRADIO INTERFACE
#  light theme using custom CSS and Soft theme
with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="orange")) as demo:
    gr.Markdown("Raw ASR Normalizer")

    with gr.Tabs():
        # TAB 1: Speech to Text
        with gr.TabItem("Audio Normalizer"):
            with gr.Row():
                with gr.Column():
                    audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Record/Upload")
                    btn_audio = gr.Button("Process Audio", variant="primary")
                with gr.Column():
                    raw_out = gr.Textbox(label="Raw ASR Output", lines=3)
                    clean_out = gr.Textbox(label="Corrected Text", lines=3)

            btn_audio.click(process_audio_pipeline, inputs=audio_input, outputs=[raw_out, clean_out])

        # TAB 2: Manual Text Input
        with gr.TabItem("Direct Text Input"):
            gr.Markdown("### Enter Noisy Hindi Text manually to clean it")
            with gr.Row():
                with gr.Column():
                    text_input = gr.Textbox(label="Input Noisy Text", placeholder="e.g., में वहा जा रहा हु...", lines=5)
                    btn_text = gr.Button("Normalize Text", variant="primary")
                with gr.Column():
                    text_output = gr.Textbox(label="Corrected Output", lines=5)

            btn_text.click(manual_text_normalize, inputs=text_input, outputs=text_output)

    gr.Markdown("---")
    gr.Markdown("Built with IndicBART & Google ASR Engine")

# Launch
demo.launch(debug=True, share=True)

/tmp/ipython-input-147076030.py:46: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="orange")) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://314ae09e91c557f2c4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
if 'demo' in locals():
    demo.close()

Closing server running on port: 7860
